In [142]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [143]:
path = "../dataset/AzureFunctionsInvocationTraceForTwoWeeksJan2021.txt"

In [144]:
df = pd.read_csv(path)

In [145]:
df['arrival_time'] = df['end_timestamp'] - df['duration']
df.sort_values(by=['arrival_time'], inplace=True, ascending=True)
df['arrival_time_fix'] = df['arrival_time'] - df['arrival_time'].iloc[0]
df.sort_values(by=['arrival_time_fix'], inplace=True, ascending=True)
df

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix
0,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,e3cdb48830f66eb8689cc0223514569a69812b77e6611e...,7.949090e-02,0.078,1.490900e-03,0.000000e+00
1,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,337cd24a7d5fd5c92460faee4ebe6a186a0eb322bd17b7...,5.715786e+01,57.154,3.860041e-03,2.369141e-03
2,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,48cc770d590d3c5a7691b3b4e9302f82ec3be5ddc2a037...,5.913048e+01,59.125,5.477905e-03,3.987005e-03
3,f274d71de386ccc77e4ca74766dbc485461c3053059d47...,3d2aee54a133509f16fb636d74128c2adcfcac71c6dcef...,6.252541e+00,6.236,1.654107e-02,1.505017e-02
4,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,68bbfd828223a505d7917339f4656c5f33ff93225cdb9d...,6.682396e-02,0.050,1.682396e-02,1.533306e-02
...,...,...,...,...,...,...
1980946,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209597e+06,0.001,1.209597e+06,1.209597e+06
1980947,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209598e+06,0.001,1.209598e+06,1.209598e+06
1980948,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,1.209599e+06
1980949,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,1.209599e+06


In [146]:
# Assuming 'df' is your DataFrame with 'arrival_time' in seconds and 'func' columns

# Step 1: Assign 4-hour trace windows
df['trace_id'] = (df['arrival_time'] // 14400).astype(int)

# Step 2: Create 1-minute bins
df['minute_bin'] = (df['arrival_time'] // 60).astype(int)

# Step 3: Count invocations per minute per trace_id and func
grouped = df.groupby(['trace_id', 'func', 'minute_bin']).size().reset_index(name='invocation_count')

# Step 4: Ensure all 240 minutes are present for each trace_id and func
filled_counts = []

for (trace_id, func), group in grouped.groupby(['trace_id', 'func']):
    minute_range = range(trace_id * 240, (trace_id + 1) * 240)
    all_minutes = pd.DataFrame({'minute_bin': minute_range})
    all_minutes['trace_id'] = trace_id
    all_minutes['func'] = func
    merged = all_minutes.merge(group, on=['trace_id', 'func', 'minute_bin'], how='left').fillna(0)
    filled_counts.append(merged)

counts_filled = pd.concat(filled_counts, ignore_index=True)

# Step 5: Compute CoV for each trace_id and func
def compute_cov(group):
    mean = group['invocation_count'].mean()
    std = group['invocation_count'].std()
    cov = std / mean if mean > 0 else np.nan
    return pd.Series({'mean': mean, 'std': std, 'cov': cov})

cov_df = counts_filled.groupby(['trace_id', 'func']).apply(compute_cov).reset_index()

# Step 6: Classify patterns
def classify_pattern(cov):
    if pd.isna(cov):
        return 'Unknown'
    elif cov < 1:
        return 'Predictable'
    elif cov < 4:
        return 'Normal'
    else:
        return 'Bursty'

cov_df['pattern'] = cov_df['cov'].apply(classify_pattern)
# # Count how many times each pattern appears
# # Count patterns per trace_id
pattern_counts_by_trace = cov_df.groupby(['trace_id', 'pattern']).size().reset_index(name='pattern_count')

# # # Merge back into cov_df
cov_df = cov_df.merge(pattern_counts_by_trace, on=['trace_id', 'pattern'], how='left')


/tmp/ipykernel_2024014/272604705.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cov_df = counts_filled.groupby(['trace_id', 'func']).apply(compute_cov).reset_index()


In [147]:
cov_df

,trace_id,func,mean,std,cov,pattern,pattern_count
0,0,04bf55f0a9d790ed3fd18c9960facdc85bca4e37974ccf...,0.041667,0.200244,4.805854,Bursty,27
1,0,09f931e5da7db2443fe669898e6074ddd6c6cc943f9880...,0.187500,0.411968,2.197162,Normal,27
2,0,0eb88b877c658e7ffe86aa9272bf6c54f2a9791c8d10b4...,0.116667,0.321694,2.757373,Normal,27
3,0,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,37.362500,49.963814,1.337272,Normal,27
4,0,26b710de36764f7be6eec2b12a979b9817916ab3397fc5...,0.004167,0.064550,15.491933,Bursty,27
...,...,...,...,...,...,...,...
6216,83,ebfe951bf36afd91115e54adab000c6325a83146c5f14e...,0.200000,0.400836,2.004180,Normal,35
6217,83,f36070dc919886d064b13a96bd0ad86ef36231cdb0f300...,0.175000,0.380761,2.175778,Normal,35
6218,83,f59e480aa82338034326aebcef6c9c43cb63e5848f98c2...,0.500000,0.501045,1.002090,Normal,35
6219,83,f5b7a048365a1ee797a55074a5b7145d285623463d3fcd...,0.500000,0.501045,1.002090,Normal,35


In [148]:
top_func_names = df[df['trace_id'] == 0]['func'].value_counts().head(7).index.tolist()
top_func_names

['155e47f8e7f751d0c845049456d01832013c61336a8cd85901330ac821a71534',
 '31aa5ab69d7730086b08f4303aa2ac3244c639866908fc3412bdfdc79ba39c22',
 'b74c4ab0e0a6349700abbf5ca5f97d54005710f2289f96a42df946539971c7f3',
 '426930ee2324a425c25463ff35403f47f2f86624df705f5cc0fde7694d855c42',
 '9bc86d6cd1ee254aaa313492f0fd88be8bd7b92d50d4237ff52d7685440c0906',
 '313c03f53a0d31f70aec25f62efb33e7dd779725ca4af579018452d1204beaad',
 '556ccf8758c8c2a20082c161e955405e950439f0503522fe129e709a5dc0e58f']

In [149]:
filtered_df = df[(df['trace_id'] == 0) & (df['func'].isin(top_func_names))]
# Lookup with tuple values
lookup = {row['func']: (row['pattern'], row['cov'], row['pattern_count']) for _, row in cov_df.iterrows()}

# Correct way to access tuple elements by index
filtered_df['pattern'] = filtered_df['func'].map(lambda f: lookup.get(f, (None, None))[0])
filtered_df['cov'] = filtered_df['func'].map(lambda f: lookup.get(f, (None, None))[1])
filtered_df['count'] = filtered_df['func'].map(lambda f: lookup.get(f, (None, None))[2])
filtered_df = filtered_df[filtered_df['pattern'] == 'Normal']
filtered_df

/tmp/ipykernel_2024014/639637826.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['pattern'] = filtered_df['func'].map(lambda f: lookup.get(f, (None, None))[0])
/tmp/ipykernel_2024014/639637826.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['cov'] = filtered_df['func'].map(lambda f: lookup.get(f, (None, None))[1])
/tmp/ipykernel_2024014/639637826.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix,trace_id,minute_bin,pattern,cov,count
24,734272c01926d19690e5ec308bab64ef97950b75b1c758...,556ccf8758c8c2a20082c161e955405e950439f0503522...,420.325099,404.987,15.338099,15.336608,0,0,Bursty,7.196250,42
25,734272c01926d19690e5ec308bab64ef97950b75b1c758...,556ccf8758c8c2a20082c161e955405e950439f0503522...,97.264052,81.921,15.343052,15.341561,0,0,Bursty,7.196250,42
26,734272c01926d19690e5ec308bab64ef97950b75b1c758...,556ccf8758c8c2a20082c161e955405e950439f0503522...,138.195055,122.852,15.343055,15.341564,0,0,Bursty,7.196250,42
27,734272c01926d19690e5ec308bab64ef97950b75b1c758...,556ccf8758c8c2a20082c161e955405e950439f0503522...,344.647737,329.304,15.343737,15.342246,0,0,Bursty,7.196250,42
28,734272c01926d19690e5ec308bab64ef97950b75b1c758...,556ccf8758c8c2a20082c161e955405e950439f0503522...,17.298993,1.955,15.343993,15.342502,0,0,Bursty,7.196250,42
...,...,...,...,...,...,...,...,...,...,...,...
16967,06da275043bac5526d5c2252a4daa222bb062165977f11...,426930ee2324a425c25463ff35403f47f2f86624df705f...,14204.756219,0.546,14204.210219,14204.208728,0,236,Bursty,4.051189,50
17070,06da275043bac5526d5c2252a4daa222bb062165977f11...,426930ee2324a425c25463ff35403f47f2f86624df705f...,14322.308388,0.453,14321.855388,14321.853897,0,238,Bursty,4.051189,50
17139,06da275043bac5526d5c2252a4daa222bb062165977f11...,426930ee2324a425c25463ff35403f47f2f86624df705f...,14370.173737,0.485,14369.688737,14369.687246,0,239,Bursty,4.051189,50
17140,06da275043bac5526d5c2252a4daa222bb062165977f11...,426930ee2324a425c25463ff35403f47f2f86624df705f...,14370.423809,0.391,14370.032809,14370.031318,0,239,Bursty,4.051189,50


In [99]:
filtered_df = cov_df[
    (cov_df['trace_id'] == 0) &
    (cov_df['pattern'].str.contains('Bursty', case=False, na=False))
]
filtered_df

,trace_id,func,mean,std,cov,pattern,pattern_count
0,0,04bf55f0a9d790ed3fd18c9960facdc85bca4e37974ccf...,0.041667,0.200244,4.805854,Bursty,27
4,0,26b710de36764f7be6eec2b12a979b9817916ab3397fc5...,0.004167,0.064550,15.491933,Bursty,27
5,0,2cf25c6b121c99858da49cc401d57da49e0bdecd64d87a...,0.012500,0.111335,8.906770,Bursty,27
8,0,324b066a00beb334a11097adf27dfa12a8493e49b72858...,0.029167,0.168625,5.781435,Bursty,27
9,0,337cd24a7d5fd5c92460faee4ebe6a186a0eb322bd17b7...,0.045833,0.209561,4.572230,Bursty,27
11,0,38efaba861a5b15a20e34eca56eb09b983b316315d47c3...,0.016667,0.128287,7.697198,Bursty,27
14,0,449a838aea8cfe82bbf555476c04178732c351ca433788...,0.033333,0.179881,5.396419,Bursty,27
15,0,481738b2980a8671d15e50f9f3e699cd205ef01ad4df49...,0.133333,0.719523,5.396419,Bursty,27
19,0,4d0668e4dc51e885b3cbd1d1bc73be7bc81734cb91ea8b...,0.162500,0.849963,5.230542,Bursty,27
21,0,514a9bcff07d21b5580057748e55e78bb062e3001a01b7...,0.033333,0.179881,5.396419,Bursty,27


In [47]:
# Step 6: Classify the invocation pattern
def classify_pattern(cov):
    if pd.isna(cov):
        return 'Unknown'
    elif cov < 1:
        return 'Predictable'
    elif cov < 4:
        return 'Normal'
    else:
        return 'Bursty'

cov_df['pattern'] = cov_df['cov'].apply(classify_pattern)


In [48]:
cov_df

,trace_id,func,mean,std,cov,pattern
0,0,04bf55f0a9d790ed3fd18c9960facdc85bca4e37974ccf...,1.000000,0.000000,0.000000,Predictable
1,0,09f931e5da7db2443fe669898e6074ddd6c6cc943f9880...,6.428571,2.878492,0.447765,Predictable
2,0,0eb88b877c658e7ffe86aa9272bf6c54f2a9791c8d10b4...,2.000000,0.000000,0.000000,Predictable
3,0,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,689.769231,393.247622,0.570115,Predictable
4,0,26b710de36764f7be6eec2b12a979b9817916ab3397fc5...,1.000000,NaN,NaN,Unknown
...,...,...,...,...,...,...
6216,83,ebfe951bf36afd91115e54adab000c6325a83146c5f14e...,2.000000,0.000000,0.000000,Predictable
6217,83,f36070dc919886d064b13a96bd0ad86ef36231cdb0f300...,3.230769,0.599145,0.185450,Predictable
6218,83,f59e480aa82338034326aebcef6c9c43cb63e5848f98c2...,5.000000,0.000000,0.000000,Predictable
6219,83,f5b7a048365a1ee797a55074a5b7145d285623463d3fcd...,5.000000,0.000000,0.000000,Predictable


In [49]:
bursty_df = cov_df[cov_df['pattern'] == 'Bursty']
bursty_df

,trace_id,func,mean,std,cov,pattern


,trace_id,func,mean,std,cov,pattern


In [25]:
bu

NameError: name 'bu' is not defined